In [12]:
from groq import Groq


client = Groq(api_key="YOUR_GROQ_API_KEY")

completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
      {
        "role": "user",
        "content": "helllo"
      }
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
)

for chunk in completion:
    print(chunk.choices[0].delta.content or "", end="")


Hello! 👋 How can I help you today?

In [13]:
# prompt chaining means multiple LLM calls

from typing import TypedDict
class BlogState(TypedDict):

    title : str
    outline : str
    content : str

In [14]:
def create_outline(state : BlogState) -> BlogState:

    title = state['title']

    prompt = f'Generate an outline for a blog on the topic - {title}'
    
    completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
    )

    answer = ''

    for chunk in completion:
        answer += chunk.choices[0].delta.content or ""
    
    state['outline'] = answer

    return state


def create_content(state : BlogState) -> BlogState:

    title = state['title']
    outline = state['outline']

    prompt = f'this is the blog title {title} and this is the outline {outline}'


    completion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
      {
        "role": "user",
        "content": prompt
      }
    ],
    temperature=1,
    max_completion_tokens=8192,
    top_p=1,
    reasoning_effort="medium",
    stream=True,
    stop=None
    )

    answer = ''

    for chunk in completion:
        answer += chunk.choices[0].delta.content or ""

    state['content'] = answer

    


In [15]:
from langgraph.graph import StateGraph , START , END

graph = StateGraph(BlogState)

graph.add_node('create_outline' , create_outline)
graph.add_node('create_content' , create_content)

In [16]:
graph.add_edge(START , 'create_outline')
graph.add_edge('create_outline' , 'create_content')
graph.add_edge('create_content' , END)

In [18]:
workflow = graph.compile()

In [19]:
initial_state = {'title' : 'Rise of AI in India'}

In [20]:
final_state = workflow.invoke(initial_state)

In [21]:
final_state

{'title': 'Rise of AI in India',
 'outline': '## Blog Outline  \n**Title: “Rise of AI in India – From Start‑ups to Policy‑Driven Growth”**\n\n| Section | Purpose | Key Points / Sub‑topics |\n|---------|---------|------------------------|\n| **1. Opening Hook** | Capture reader’s curiosity with a striking fact or anecdote. | • 2023: India ranked 5th globally in AI research output.<br>• A single AI‑driven health app saved 1,200 lives in rural Uttar Pradesh. |\n| **2. Why AI? The Context in India** | Set the stage: why AI is a national priority. | • Rapid digital adoption (3.1B internet users).<br>• Demographic dividend – 60% under 30. <br>• Need for scalable solutions in public services. |\n| **3. Government Initiatives & Policy Framework** | Highlight top‑down support. | • National AI Strategy 2022 – “AI for All” vision. <br>• AI Policy & National AI Mission (AI Mission). <br>• AI-focused funding: IARPA‑India, NITI Aayog’s AI Fund. <br>• Digital India & Smart Cities: AI as core. |\n| **

In [22]:
final_state['outline']

'## Blog Outline  \n**Title: “Rise of AI in India – From Start‑ups to Policy‑Driven Growth”**\n\n| Section | Purpose | Key Points / Sub‑topics |\n|---------|---------|------------------------|\n| **1. Opening Hook** | Capture reader’s curiosity with a striking fact or anecdote. | • 2023: India ranked 5th globally in AI research output.<br>• A single AI‑driven health app saved 1,200 lives in rural Uttar Pradesh. |\n| **2. Why AI? The Context in India** | Set the stage: why AI is a national priority. | • Rapid digital adoption (3.1B internet users).<br>• Demographic dividend – 60% under 30. <br>• Need for scalable solutions in public services. |\n| **3. Government Initiatives & Policy Framework** | Highlight top‑down support. | • National AI Strategy 2022 – “AI for All” vision. <br>• AI Policy & National AI Mission (AI Mission). <br>• AI-focused funding: IARPA‑India, NITI Aayog’s AI Fund. <br>• Digital India & Smart Cities: AI as core. |\n| **4. Ecosystem Players** | Showcase the key sta